## GateMask deep dive — learning a sparse explanation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/khairulislam/tslens/blob/main/notebooks/gatemask.ipynb)

**GateMask**, from ContraLSP, is tslens's only learned-mask method. We compare it with WinTSR on the exact synthetic task from the quickstart, where the true signal is known. GateMask trains a new mask model for every attribution call, so this is naturally the slowest notebook in the series, but it still runs on free Colab CPU in a few minutes.

Papers: [ContraLSP, ICLR 2024](https://arxiv.org/abs/2401.08552) · [WinTSR](https://arxiv.org/abs/2412.04532)

In [ ]:
%pip install -q tslens matplotlib pytorch-lightning

## 1. A dataset where we know the right answer

We build 5 autocorrelated (AR(1)) input channels of length 50. The target depends on
**one feature (0), over one window (time steps 20–25)**. Everything else is noise.

That known window is our ground truth: a good interpretability method should light up
there and nowhere else.

In [ ]:
import numpy as np
import torch
from torch import nn

torch.manual_seed(0)

SEQ_LEN, N_FEATURES = 50, 5
SIGNAL_FEATURE, SIGNAL_START, SIGNAL_END = 0, 20, 26
RHO = 0.9  # autocorrelation: real time series are not i.i.d. noise


def ar_series(n, rho=RHO):
    eps = torch.randn(n, SEQ_LEN, N_FEATURES)
    x = torch.empty_like(eps)
    x[:, 0] = eps[:, 0]
    for t in range(1, SEQ_LEN):
        x[:, t] = rho * x[:, t - 1] + (1 - rho ** 2) ** 0.5 * eps[:, t]
    return x


def make(n):
    x = ar_series(n)
    y = x[:, SIGNAL_START:SIGNAL_END, SIGNAL_FEATURE].sum(dim=1, keepdim=True)
    return x, y


x_train, y_train = make(4000)
x_test, y_test = make(256)

ground_truth = torch.zeros(SEQ_LEN, N_FEATURES)
ground_truth[SIGNAL_START:SIGNAL_END, SIGNAL_FEATURE] = 1

print("inputs ", tuple(x_train.shape), " targets", tuple(y_train.shape))
print(f"ground truth: feature {SIGNAL_FEATURE}, steps {SIGNAL_START}-{SIGNAL_END - 1}")

## 2. Train a small GRU

About 20 seconds on CPU.

In [ ]:
class GRUForecaster(nn.Module):
    def __init__(self, hidden=64):
        super().__init__()
        self.gru = nn.GRU(N_FEATURES, hidden, batch_first=True)
        self.head = nn.Linear(hidden, 1)

    def forward(self, x):
        out, _ = self.gru(x)
        return self.head(out.mean(dim=1))


model = GRUForecaster()
opt = torch.optim.Adam(model.parameters(), lr=3e-3)
loss_fn = nn.MSELoss()

for epoch in range(25):
    perm = torch.randperm(len(x_train))
    for i in range(0, len(x_train), 128):
        idx = perm[i : i + 128]
        opt.zero_grad()
        loss_fn(model(x_train[idx]), y_train[idx]).backward()
        opt.step()

model.eval()
with torch.no_grad():
    r2 = 1 - loss_fn(model(x_test), y_test).item() / y_test.var().item()
print(f"test R2 = {r2:.4f}   (needs to be high, or there is no signal to explain)")

## 3. What makes GateMask different

WinTSR, WinIT, Occlusion, and Integrated Gradients compute scores directly from perturbations or gradients on the input being explained. GateMask instead **trains a small neural network on every call to `.attribute()`**. Its job is to produce a sparse, smooth, binary-skewed gate over `(time, feature)` while keeping the masked input's prediction close to the original. ContraLSP combines counterfactual perturbation with contrastive learning to keep perturbations in distribution.

That optimization is why GateMask needs a `pytorch_lightning.Trainer` and `batch_size`, why `max_epochs` affects explanation quality, and why it is markedly slower. `representation()` returns the learned gate itself with shape `(batch, seq_len, n_features)`: unlike WinTSR and WinIT, GateMask has no explicit `n_output` axis because one mask network is fitted jointly across the batch and the model output. GateMask accepts only one attributed input tensor; extra model context belongs in `additional_forward_args`.

## 4. Fit GateMask and compare its runtime

Both methods explain the same 16 inputs against the same zero baseline. The timers make GateMask's per-call optimization cost concrete.

In [ ]:
import time

from pytorch_lightning import Trainer
from tslens import WinTSR
from tslens.attr import GateMask

inputs = x_test[:16]
baselines = torch.zeros_like(inputs)

start = time.perf_counter()
wintsr_attr = WinTSR(model).attribute(
    inputs, baselines=baselines, threshold=0.5
)
wintsr_seconds = time.perf_counter() - start

torch.manual_seed(0)
explainer = GateMask(model)
start = time.perf_counter()
attr = explainer.attribute(
    inputs=inputs,
    baselines=baselines,
    trainer=Trainer(
        max_epochs=50, accelerator="cpu", enable_progress_bar=False,
        enable_model_summary=False, logger=False,
    ),
    batch_size=16,
)
gatemask_seconds = time.perf_counter() - start

print(f"WinTSR:  {wintsr_seconds:7.2f} s  {tuple(wintsr_attr.shape)}")
print(f"GateMask: {gatemask_seconds:7.2f} s  {tuple(attr.shape)}")

## 5. Inspect the learned mask

Averaging the sample-specific masks reveals whether GateMask consistently recovers the planted feature-0 window at steps 20–25. In this seeded run the correct window is elevated, but the background remains diffuse rather than producing a perfectly binary recovery. The signal/background means below quantify that separation: a larger ratio means a cleaner mask.

In [ ]:
import matplotlib.pyplot as plt

gatemask_saliency = attr.abs().mean(dim=0)
signal_mean = gatemask_saliency[ground_truth.bool()].mean().item()
background_mean = gatemask_saliency[~ground_truth.bool()].mean().item()
print(f"signal mean: {signal_mean:.3f}  background mean: {background_mean:.3f}  ratio: {signal_mean / background_mean:.2f}x")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(ground_truth.T, aspect="auto", cmap="Greys", vmin=0, vmax=1)
axes[0].set_title("Ground truth")
axes[1].imshow(gatemask_saliency.T, aspect="auto", cmap="viridis", vmin=0, vmax=1)
axes[1].set_title("GateMask — 50 epochs")

for ax in axes:
    ax.set_xlabel("time step")
    ax.set_ylabel("feature")
    ax.set_yticks(range(N_FEATURES))

plt.tight_layout()
plt.show()

## 6. `max_epochs` is a training-quality knob

Direct attribution methods do not have an inner training loop. GateMask does: too few epochs may leave a diffuse or underfit mask, while more epochs give the sparsity and prediction-preservation objectives longer to settle. The effect is empirical rather than guaranteed, so compare the masks instead of assuming that longer always means sharper. We reuse the 50-epoch fit above and train fresh 10- and 150-epoch masks with the same seed.

In [ ]:
def fit_gatemask(max_epochs):
    torch.manual_seed(0)
    return GateMask(model).attribute(
        inputs=inputs,
        baselines=baselines,
        trainer=Trainer(
            max_epochs=max_epochs, accelerator="cpu", enable_progress_bar=False,
            enable_model_summary=False, logger=False,
        ),
        batch_size=16,
    )


epoch_masks = {10: fit_gatemask(10), 50: attr, 150: fit_gatemask(150)}

print(f"{'epochs':>6}  {'signal':>8}  {'background':>10}  {'ratio':>7}")
for epochs, mask in epoch_masks.items():
    mean_mask = mask.abs().mean(dim=0)
    signal = mean_mask[ground_truth.bool()].mean().item()
    background = mean_mask[~ground_truth.bool()].mean().item()
    print(f"{epochs:>6}  {signal:>8.3f}  {background:>10.3f}  {signal / background:>6.2f}x")

fig, axes = plt.subplots(1, 3, figsize=(14, 3.2))
for ax, (epochs, mask) in zip(axes, epoch_masks.items()):
    ax.imshow(mask.abs().mean(dim=0).T, aspect="auto", cmap="viridis", vmin=0, vmax=1)
    ax.set_title(f"GateMask — {epochs} epochs")
    ax.set_xlabel("time step")
    ax.set_ylabel("feature")
    ax.set_yticks(range(N_FEATURES))

plt.tight_layout()
plt.show()

## Next steps

- Tune `max_epochs`, `win_size`, and `sigma` for the data and runtime budget.
- Use `additional_forward_args` for a single-input model that also needs padding masks or other context.
- Compare with the [WinIT deep dive](https://colab.research.google.com/github/khairulislam/tslens/blob/main/notebooks/winit.ipynb) and [WinTSR quickstart](https://colab.research.google.com/github/khairulislam/tslens/blob/main/notebooks/quickstart.ipynb).

```bibtex
@inproceedings{liu2024explaining,
  title={Explaining Time Series via Contrastive and Locally Sparse Perturbations},
  author={Liu, Zichuan and Zhang, Yingying and Wang, Tianchun and Wang, Zefan and Luo, Dongsheng and Du, Mengnan and Wu, Min and Wang, Yi and Chen, Chunlin and Fan, Lunting and Wen, Qingsong},
  booktitle={International Conference on Learning Representations},
  year={2024}
}

@article{islam2024wintsr,
  title={WinTSR: A Windowed Temporal Saliency Rescaling Method for Interpreting Time Series Deep Learning Models},
  author={Islam, Md Khairul and Fox, Judy},
  journal={arXiv preprint arXiv:2412.04532},
  year={2024}
}
```